```mermaid
flowchart LR
    A0["00"] --> A1a["01a"] --> A1b["01b"] --> A2["02"] --> A3["03"] --> A4a["04a"] --> A4b["04b"]
    A4b --> A5a["05a"] --> A5b["05b"] --> A6a["06a"] --> A6b["06b"]
    A6b --> A7["07"] --> A8a["08a"] --> A8b["08b"]
    A8b --> A9["09"] --> A10["10"] --> A11["11"] --> A12["12"] 
    
    classDef normal fill:#f8f9fa,stroke:#adb5bd,stroke-width:1px,color:#111;
    classDef done fill:#e8f7f0,stroke:#198754,stroke-width:1.5px,color:#111;
    classDef current fill:#fff3cd,stroke:#ff8c00,stroke-width:2px,color:#111;
    
    class A0,A1a,A1b,A2 done;
    class A3 current;
    class A4a,A4b,A5a,A5b,A6a,A6b,A7,A8a,A8b,A9,A10,A11,A12 normal;
```

# Notebook 03 — Exploratory Corpus Analysis II: Distributions, Dispersion, and Time

**Course theme:** knowledge dynamics in philosophical texts over time.

**What this notebook does**
- Loads cleaned Project Gutenberg texts from `corpus/texts_cleaned/`.
- Loads metadata from `corpus_analysis/NLP2026-corpus-complete-metadata.csv` (fallback to `NLP2026-selected-metadata.csv`).
- Builds a document-term matrix (counts) and computes:
  - frequency distributions / long tail (Zipf-style)
  - dispersion across documents
  - time-binned trends for selected concepts
- Saves figures and tables to disk so later notebooks can load them without recomputation.

> **Method note:** This notebook prioritizes *reproducible descriptive analysis* (counts-based). Embeddings appear later as a complementary lens.

In [ ]:
!pip install scipy

In [ ]:
# -----------------------------
# Import
# -----------------------------
import re
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from tqdm.auto import tqdm
from scipy import sparse

# -----------------------------
# Paths
# -----------------------------
PROJECT_ROOT = Path(".") 

TEXTS_DIR = PROJECT_ROOT / "data" / "processed" / "cleaned"
CACHE_DIR = PROJECT_ROOT / "cache" 
OUTPUT_DIR = PROJECT_ROOT / "analysis" 

META = PROJECT_ROOT / "analysis" / "tables" / "nb02-corpus-complete-metadata.csv"
META_00 = PROJECT_ROOT / "analysis" / "tables" / "nb02-metadata.csv"

# -----------------------------
# Parameters
# -----------------------------
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print("TEXTS_DIR:", TEXTS_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("CACHE_DIR:", CACHE_DIR)

In [ ]:
# -----------------------------
# HELPER
# -----------------------------
def extract_pg_id(path: Path) -> int | None:
    """Extract Project Gutenberg numeric id from filenames like pg12345.txt."""
    m = re.search(r"pg(\d+)", path.stem.lower())
    return int(m.group(1)) if m else None

In [ ]:
# -----------------------------
# Load texts
# -----------------------------
text_paths = sorted(TEXTS_DIR.glob("pg*.txt"))
if not text_paths:
    raise FileNotFoundError(f"No pg*.txt found in {TEXTS_DIR}")

docs = pd.DataFrame({
    "pg_id": [extract_pg_id(p) for p in text_paths],
    "path": [str(p) for p in text_paths],
})
docs["filename"] = docs["path"].apply(lambda x: Path(x).name)

# Read texts (fast enough for ~500 books on CPU)
def read_text(p: str) -> str:
    return Path(p).read_text(encoding="utf-8", errors="replace")

docs["text"] = docs["path"].map(read_text)
docs["n_chars"] = docs["text"].str.len()
docs["n_lines"] = docs["text"].str.count("\n") + 1

print("Documents loaded:", len(docs))
docs.head()

In [ ]:
# -----------------------------
# Load metadata
# -----------------------------

meta = pd.read_csv(META)
print(f"\nLoaded metadata from {META} | rows={len(meta):,} cols={len(meta.columns):,}\n")
print("=" * 80)
display(meta.head(3))

In [ ]:
# -----------------------------
# Convert and merge metadata
# -----------------------------

# Convert digits to integers
meta["author_birthdate"] = pd.to_numeric(meta["author_birthdate"], errors="coerce").astype("Int64")
meta["author_deathdate"] = pd.to_numeric(meta["author_deathdate"], errors="coerce").astype("Int64")
meta["publication_year"] = pd.to_numeric(meta["publication_year"], errors="coerce").astype("Int64")

# Merge metadata and texts
df = docs.merge(meta, on="pg_id", how="left", suffixes=("", "_meta"))

# Part 1: Calculate and assign time-bins

## Time-Bins: Grouping Texts Across History

Our corpus spans roughly **2,300 years** — from around 400 BCE to the mid-20th century — but individual texts aren't evenly distributed across that timespan. Some centuries are richly represented; others are sparse. Before we can study *change over time* (e.g. semantic shift, topic evolution), we need to group texts into **time-bins**: discrete periods that let us treat "all texts published in this bin" as one snapshot to compare against other snapshots.

There are two common strategies for defining these bins, and the choice matters for what your results will actually show.

### Fixed bins (equal-width in time)

Fixed bins split the timeline into intervals of equal *duration* — for example, every 100 years, regardless of how many texts fall into each interval.

- **Pro:** intervals are directly comparable across history — "the 1700s" means the same amount of elapsed time as "the 1200s." This is the natural choice when your research question is genuinely about *historical periods* (e.g. "did the Enlightenment coincide with a shift in how 'reason' was used?").
- **Con:** because our corpus is unevenly distributed over time, some fixed bins will contain many texts and others very few (or none) — a bin with 3 texts and a bin with 80 texts are not equally reliable for estimating word usage or topic prevalence.

### Quantile bins (equal-count)

Quantile bins instead split the corpus so that each bin contains roughly the **same number of texts**, regardless of how much calendar time that requires.

```python
import pandas as pd

# Assign each document to one of N equal-sized bins based on its date
n_bins = 6
df["time_bin"] = pd.qcut(year.astype(float), q=n_bins, duplicates="drop")
```

- **Pro:** every bin has comparable statistical power — word frequencies, topic proportions, and embedding averages are estimated from a similar number of documents in each period, which matters for methods sensitive to sample size (like topic modelling or averaging embeddings).
- **Con:** bins no longer correspond to fixed, interpretable historical spans — a densely-populated era (say, 18th-century texts) might be split into several narrow bins, while a sparse era (say, Late Antiquity) gets collapsed into one wide bin covering centuries. Bin boundaries need to be reported explicitly, since "bin 3" isn't self-explanatory the way "the 1600s" is.

### Which to use

- Use **quantile bins** when you want statistically balanced comparisons across time (e.g. detecting semantic drift with embeddings, where uneven sample sizes per period can bias the result).
- Use **fixed bins** when your question is tied to actual historical periods and calendar time itself is meaningful (e.g. comparing "before vs. after" a known intellectual turning point).

Both are valid — the key is to be explicit in your notebook about which strategy you used and why, since it directly shapes how any temporal pattern you find should be interpreted.

In [ ]:
def time_bins(year: pd.Series) -> pd.Categorical:
    y = year.dropna().astype(int)
    if y.empty:
        raise ValueError("No valid years found to create time bins.")
    lo, hi = int(y.min()), int(y.max())
    if TIME_BIN_MODE == "fixed":
        start = (lo // FIXED_BIN_WIDTH) * FIXED_BIN_WIDTH
        end = ((hi // FIXED_BIN_WIDTH) + 1) * FIXED_BIN_WIDTH
        bins = list(range(start, end + 1, FIXED_BIN_WIDTH))
        labels = [f"{bins[i]}–{bins[i+1]-1}" for i in range(len(bins)-1)]
        binned = pd.cut(year.astype(float), bins=bins, labels=labels, include_lowest=True)
        return binned
    elif TIME_BIN_MODE == "quantile":
        binned = pd.qcut(year.astype(float), q=N_QUANTILE_BINS, duplicates="drop")
        
        # Rename interval labels to integer ranges (display only)
        if hasattr(binned, "cat"):
            new_labels = [
                f"{int(round(iv.left))}–{int(round(iv.right))}"
                for iv in binned.cat.categories
            ]
            binned = binned.cat.rename_categories(new_labels)
        
        return binned
    else:
        raise ValueError("TIME_BIN_MODE must be 'fixed' or 'quantile'.")

In [ ]:
# =============================================== YOUR CODE HERE ===============================================
# Time binning choices
TIME_BIN_MODE = ""         # Decide which mode you want to use: "fixed" or "quantile" 
FIXED_BIN_WIDTH =          # If TIME_BIN_MODE="fixed", decide how many years should be in each bin (e.g., 50-year bins)
N_QUANTILE_BINS =          # if TIME_BIN_MODE="quantile", set how many bins will be used to divide the period

# Create time bins
df["time_bin"] = time_bins(df["publication_year"])

display(df.head(3))

print("=" * 80)
print("\nYear coverage:", df["publication_year"].notna().mean().round(3))
print("=" * 80)


In [ ]:
# -----------------------------
# Visualise coverage over time
# -----------------------------
bin_counts = df["time_bin"].value_counts(dropna=False).sort_index()
length_by_bin = df.groupby("time_bin", observed=True)["n_chars"].mean()

fig, axes = plt.subplots(2, 1, figsize=(11, 8), sharex=True)

bin_counts.plot(kind="bar", ax=axes[0], color='teal')
axes[0].set_title("Documents per time bin")
axes[0].set_ylabel("# documents")

length_by_bin.plot(kind="bar", ax=axes[1], color='gold')
axes[1].set_title("Average document length (chars) per time bin")
axes[1].set_ylabel("mean chars")

plt.tight_layout()
fig_path = OUTPUT_DIR / "figures" / "nb03-timebin_sampling_artifacts.png"
plt.savefig(fig_path, dpi=200)
print("\nSaved:", fig_path.name)
plt.show()

In [ ]:
bin_counts.to_csv(OUTPUT_DIR / "tables" / "nb03-timebin_doc_counts.csv", header=["n_docs"])
length_by_bin.to_csv(OUTPUT_DIR / "tables" / "nb03-timebin_mean_chars.csv", header=["mean_chars"])
print(f"Saved timebin doc counts to: {OUTPUT_DIR / "tables" / "nb03-timebin_doc_counts.csv"}")
print(f"Saved timebin mean chars to: {OUTPUT_DIR / "tables" / "nb03-timebin_mean_chars.csv"}")

# Part 2: The Document-Term Matrix

Before a computer can analyze a text corpus, we need to turn words into numbers. The simplest way to do this is the **document-term matrix (DTM)**.

**The idea**

We represent the corpus as a table:

- Each **row** is one document (in our case, one philosophical text).
- Each **column** is one distinct word (a "term") found anywhere in the corpus.
- Each **cell** contains a count: how many times that word appears in that document.

|              | reason | nature | soul | ... |
|--------------|--------|--------|------|-----|
| Text 1       | 12     | 3      | 0    | ... |
| Text 2       | 0      | 7      | 15   | ... |
| Text 3       | 5      | 5      | 2    | ... |

This is called a **"bag of words"** representation: it only tracks *how often* each word appears, completely ignoring word order, grammar, and sentence structure. "Reason follows nature" and "nature follows reason" would produce identical rows.

**Why it is "sparse"**

With 572 texts and a vocabulary that could easily reach tens of thousands of distinct words, most documents only use a small fraction of the total vocabulary. This means most cells in the matrix are 0 — a document about ethics will have plenty of zeros in columns for terms that only appear in texts on logic or metaphysics. This is what we mean when we call the matrix **sparse**.

**Building it with `CountVectorizer`**

`CountVectorizer` (from `scikit-learn`) automates this process: it scans the corpus, builds a vocabulary of all distinct terms, and produces the matrix of counts.

```python
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer()
dtm = vectorizer.fit_transform(corpus)  # corpus = list of texts

# Inspect the vocabulary and matrix shape
print(vectorizer.get_feature_names_out()[:10])
print(dtm.shape)  # (n_documents, n_terms)
```

**Why this matters for our project**

The document-term matrix is the starting point for almost everything that follows this session — it is the raw material that TF-IDF, topic modelling, and classification all build on. Raw counts are a useful first step, but they have a well-known limitation: frequent function words (*the*, *is*, *and*) dominate the matrix even though they carry little meaning. This is exactly the problem the next representations we will cover (TF-IDF, embeddings) are designed to solve.

In [ ]:
# -----------------------------
# Build / load document-term matrix (counts)
# -----------------------------
from sklearn.feature_extraction.text import CountVectorizer
import joblib

# Analysis parameters
MAX_FEATURES = 50000          # keep CPU/RAM stable
MIN_DF = 3                    # ignore extremely rare terms
NGRAM_RANGE = (1, 2)          # unigrams + bigrams (optional but useful for concepts)
LOWERCASE = True

# Save to 'cache' recomputable intermediate artifacts that make notebooks faster
X_path = CACHE_DIR / "nb03-X_counts.npz"
vocab_path = CACHE_DIR / "nb03-vocab.json"
vectorizer_path = CACHE_DIR / "nb03-vectorizer.joblib"

# Save to 'analysis' the tables that can be used in future notebooks
doc_index_path = OUTPUT_DIR / "tables" / "nb03-doc_index.csv"

if X_path.exists() and vocab_path.exists() and doc_index_path.exists():
    X = sparse.load_npz(X_path)
    with open(vocab_path, "r", encoding="utf-8") as f:
        vocab = json.load(f)
    df_index = pd.read_csv(doc_index_path)
    print(f"Loaded cached DTM: {X.shape} from {X_path.name}")
else:
    vectorizer = CountVectorizer(
        lowercase=LOWERCASE,
        stop_words="english",
        min_df=MIN_DF,
        max_features=MAX_FEATURES,
        ngram_range=NGRAM_RANGE,
        token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z'-]{2,}\b",
    )
    
    # Add progress bar: tqdm wraps the list of texts and shows progress
    texts = df["text"].tolist()
    print("Building document-term matrix...")
    X = vectorizer.fit_transform(tqdm(texts, desc="Processing texts"))
    
    vocab = {v: int(i) for v, i in vectorizer.vocabulary_.items()}

    sparse.save_npz(X_path, X)
    with open(vocab_path, "w", encoding="utf-8") as f:
        json.dump(vocab, f)
    joblib.dump(vectorizer, vectorizer_path)
    df[["pg_id", "title", "publication_year", "time_bin", "n_chars"]].to_csv(doc_index_path, index=False)

    print(f"Built DTM: {X.shape} and saved to {CACHE_DIR}")

# Fill the gap

In [ ]:
# =============================================== YOUR CODE HERE ===============================================
# Print the matrix shape


# Rank-frequency analysis (Zipf's law)

This block computes the raw term frequencies from the document-term matrix,
ranks the terms by frequency, and produces two plots:
  1. A horizontal bar chart of the top N most frequent terms.
  2. A log-log scatter plot of frequency vs. rank to visualise the long-tail
     distribution and check whether the corpus approximately follows Zipf's law
     (straight line with slope ~ -1).

The rank-frequency plot helps to understand lexical diversity and the dominance
of a small set of high-frequency terms. It is a diagnostic tool for corpus
characterisation and also informs downstream choices (e.g., whether to filter
by max_features or to use tf‑idf weighting).

In [ ]:
# -----------------------------
# Frequency distributions (long tail)
# -----------------------------

# Load vectorizer if available to get feature names in correct order
vectorizer = None
if vectorizer_path.exists():
    vectorizer = joblib.load(vectorizer_path)
    terms = vectorizer.get_feature_names_out()
else:
    # fallback: build array of terms by inverting vocab
    inv = {i: t for t, i in vocab.items()}
    terms = np.array([inv[i] for i in range(len(inv))])

tf = np.asarray(X.sum(axis=0)).ravel()
df_terms = pd.DataFrame({"term": terms, "tf": tf})
df_terms = df_terms.sort_values("tf", ascending=False).reset_index(drop=True)
df_terms["rank"] = np.arange(1, len(df_terms) + 1)

# Top terms
top_n = 30
fig, ax = plt.subplots(figsize=(11, 6))
df_terms.head(top_n).iloc[::-1].plot(kind="barh", x="term", y="tf", ax=ax, legend=False, color='teal')
ax.set_title(f"Top {top_n} terms by corpus frequency (after stopwords)")
ax.set_xlabel("term frequency")
plt.tight_layout()
fig_path = OUTPUT_DIR / "figures" / "nb03-top_terms_barh.png"
plt.savefig(fig_path, dpi=200)
plt.show()

# Zipf-style plot
fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(df_terms["rank"], df_terms["tf"], color='teal')
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_title("Rank–frequency (log–log): long tail")
ax.set_xlabel("rank (log)")
ax.set_ylabel("term frequency (log)")
plt.tight_layout()
fig_path2 = OUTPUT_DIR / "figures" / "nb03-zipf_rank_frequency.png"
plt.savefig(fig_path2, dpi=200)
plt.show()

df_terms.head(10)

In [ ]:
# Save term table (trimmed)
df_terms.head(5000).to_csv(OUTPUT_DIR / "tables" / "nb03-term_frequencies_top5000.csv", index=False)
df_terms.head(200).to_csv(OUTPUT_DIR / "tables" / "nb03-term_frequencies_top200.csv", index=False)
print("Saved term frequency tables to:", OUTPUT_DIR / "tables")

# Dispersion and time trends for selected concepts

In [ ]:
def get_term_index(term: str) -> int:
    """
    Retrieve the vocabulary index for a term (case‑insensitive).

    Parameters: term(str)

    Returns: int or None
        The index of the term in the global `vocab_map`, or `None` if not found.

    """
    term = term.lower().strip()
    return vocab_map.get(term)

# Fill the gap

In [ ]:
# =============================================== YOUR CODE HERE ===============================================
# Edit the list of concepts to track
CONCEPTS = [
]

# Map terms to column indices
vocab_map = vectorizer.vocabulary_ if vectorizer is not None else vocab

concept_rows = []
# =============================================== YOUR CODE HERE ===============================================
n_docs = X.shape[0] # Extract from the shape of our DTM the number of documents

for c in CONCEPTS:
    idx = get_term_index(c)
    if idx is None:
        concept_rows.append({"concept": c, "in_vocab": False, "doc_freq": 0, "doc_prop": 0.0})
        continue
    col = X[:, idx]
    doc_freq = int((col > 0).sum())
    concept_rows.append({
        "concept": c,
        "in_vocab": True,
        "doc_freq": doc_freq,
        "doc_prop": doc_freq / n_docs
    })

df_concepts = pd.DataFrame(concept_rows).sort_values(["in_vocab", "doc_prop"], ascending=[False, False])
df_concepts.to_csv(OUTPUT_DIR / "tables" / "nb03-concept_dispersion_docfreq.csv", index=False)
df_concepts

In [ ]:
# -----------------------------
# Time-binned normalized frequencies (per million tokens)
# -----------------------------

valid = df["time_bin"].notna()
df_valid = df.loc[valid].copy()
X_valid = X[valid.values, :]

bins = df_valid["time_bin"].astype(str)

# ----- FIX: sort chronologically by start year -----
def get_start_year(b: str) -> int:
    try:
        return int(b.split('–')[0])
    except (ValueError, IndexError, AttributeError):
        return 0

bin_levels = sorted(bins.unique(), key=get_start_year)

# Precompute totals per bin (tokens in our DTM space)
bin_total_tokens = {}
bin_doc_counts = {}
for b in bin_levels:
    mask = (bins == b).values
    Xb = X_valid[mask, :]
    bin_total_tokens[b] = float(Xb.sum())
    bin_doc_counts[b] = int(mask.sum())

rows = []
for c in CONCEPTS:
    idx = get_term_index(c)
    for b in bin_levels:
        if idx is None:
            rows.append({"concept": c, "time_bin": b, "count": 0.0, "per_million": np.nan})
            continue
        mask = (bins == b).values
        Xb = X_valid[mask, :]
        cnt = float(Xb[:, idx].sum())
        denom = bin_total_tokens[b] if bin_total_tokens[b] > 0 else np.nan
        rows.append({
            "concept": c,
            "time_bin": b,
            "count": cnt,
            "per_million": (cnt / denom) * 1e6 if denom and not np.isnan(denom) else np.nan
        })

df_trends = pd.DataFrame(rows)
df_trends.to_csv(OUTPUT_DIR / "tables" / "nb03-concept_trends_per_million.csv", index=False)

# Plot trends for concepts that are in vocab
in_vocab = df_concepts.query("in_vocab == True")["concept"].tolist()
plot_df = df_trends[df_trends["concept"].isin(in_vocab)].copy()
plot_df["time_bin"] = pd.Categorical(plot_df["time_bin"], categories=bin_levels, ordered=True)

fig, ax = plt.subplots(figsize=(12, 6))
for c in in_vocab:
    sub = plot_df[plot_df["concept"] == c].sort_values("time_bin")
    ax.plot(sub["time_bin"].astype(str), sub["per_million"], marker="o", label=c)

ax.set_title("Concept prevalence over time (normalized per million tokens)")
ax.set_xlabel("time bin")
ax.set_ylabel("per million tokens (DTM space)")
ax.legend(ncol=2, fontsize=9)
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
fig_path = OUTPUT_DIR / "figures" / "nb03-concept_trends_over_time.png"
plt.savefig(fig_path, dpi=200)
plt.show()

print("Saved:", fig_path.name)

In [ ]:
# -----------------------------
# Heatmap: concepts x time bins
# -----------------------------

# Get all unique time bins and sort chronologically
all_bins = df_trends['time_bin'].unique()
chronological_bins = sorted(all_bins, key=get_start_year)

heat = df_trends.copy()
heat = heat[heat["concept"].isin(in_vocab)]
pivot = heat.pivot_table(index="concept", columns="time_bin", values="per_million", aggfunc="mean")
pivot = pivot.reindex(index=in_vocab)

# Reorder columns by chronological order
pivot = pivot[chronological_bins]   # only keep bins that exist in pivot

fig, ax = plt.subplots(figsize=(12, max(3, 0.35 * len(in_vocab) + 1)))
sns.heatmap(pivot, 
            cmap="crest", 
            ax=ax)
ax.set_title("Concept prevalence heatmap (per million tokens)")
ax.set_xlabel("time bin")
ax.set_ylabel("concept")
plt.tight_layout()
fig_path = OUTPUT_DIR / "figures" / "nb03-concept_trends_heatmap.png"
plt.savefig(fig_path, dpi=200)
plt.show()

## Keyness analysis: what words distinguish early from late philosophy?

So far we've grouped texts into time-bins and represented them as word counts. Now we ask a natural question: **which words are distinctively associated with early periods, and which with late periods?**

We use a classic technique in corpus linguistics called **keyness analysis**; instead of just looking at raw frequency (which is dominated by common words like "the" or "is"), we compare *how a word's usage differs between two groups of texts*.

### The comparison: early vs. late

We collapse our many time-bins down to two groups so the comparison is simple: the earliest bins vs. the latest bins. If we have at least four time-bins, we take the first two as "early" and the last two as "late." With fewer bins, we fall back to a simpler split: everything before the median publication year is "early," everything after is "late."

### Why log-odds, not just raw counts?

A word that appears 100 times in "early" texts and 50 times in "late" texts is not necessarily twice as characteristic of "early" — it depends on how large each group is overall. To make a fair comparison, we:

1. Convert each word's raw count into a **probability**: count of the word ÷ total word count in that group. This accounts for the two groups being different sizes.
2. Take the **log-odds ratio**: `log(p_late) − log(p_early)`. This gives one number per word:
   - **Positive** → the word is proportionally more common in *late* texts.
   - **Negative** → the word is proportionally more common in *early* texts.
   - **Near zero** → the word is used similarly in both periods.

### Additive smoothing

If a word never appears in one of the two groups, its probability would be exactly 0, and `log(0)` is undefined (mathematically, negative infinity). **Additive smoothing** or **Add-1 smoothing** (also called Laplace smoothing: https://en.wikipedia.org/wiki/Additive_smoothing) fixes this by pretending every word appeared one extra time in every group before computing probabilities. This avoids division-by-zero and undefined logarithms, at the cost of slightly softening extreme results for very rare words.

### Reading the output

The code ranks all terms by their log-odds score and plots the **top 25 words most characteristic of the *late* period** (the words with the highest positive scores). The full ranked list — including words most characteristic of the *early* period, at the opposite end of the sorted table — is saved to a CSV for further inspection.

This is a lightweight, interpretable first pass at detecting **semantic and thematic change over time**, and a natural complement to the embedding-based drift methods we will use later in the course.

In [ ]:
# -----------------------------
# Keyness-style comparison: early vs late periods (log odds with add-1 smoothing)
# -----------------------------
if len(bin_levels) >= 4:
    early_bins = bin_levels[:2]
    late_bins = bin_levels[-2:]
else:
    # fallback: split by median year
    med = df_valid["year"].median()
    early_bins = sorted(df_valid.loc[df_valid["year"] <= med, "time_bin"].astype(str).unique())
    late_bins = sorted(df_valid.loc[df_valid["year"] > med, "time_bin"].astype(str).unique())

mask_early = df_valid["time_bin"].astype(str).isin(early_bins).values
mask_late = df_valid["time_bin"].astype(str).isin(late_bins).values

X_early = X_valid[mask_early, :]
X_late = X_valid[mask_late, :]

c_early = np.asarray(X_early.sum(axis=0)).ravel()
c_late = np.asarray(X_late.sum(axis=0)).ravel()

# add-1 smoothing
alpha = 1.0
p_early = (c_early + alpha) / (c_early.sum() + alpha * len(c_early))
p_late = (c_late + alpha) / (c_late.sum() + alpha * len(c_late))

log_odds = np.log(p_late) - np.log(p_early)
df_key = pd.DataFrame({"term": terms, "log_odds_late_minus_early": log_odds, "tf_early": c_early, "tf_late": c_late})
df_key = df_key.sort_values("log_odds_late_minus_early", ascending=False).reset_index(drop=True)

topk = 25
fig, ax = plt.subplots(figsize=(11, 6))
df_key.head(topk).iloc[::-1].plot(kind="barh", x="term", y="log_odds_late_minus_early", ax=ax, legend=False, color='teal')
ax.set_title(f"Terms more characteristic of LATE vs EARLY bins (log-odds; early={early_bins}, late={late_bins})")
ax.set_xlabel("log odds (late - early)")
plt.tight_layout()
fig_path = OUTPUT_DIR / "figures" / "nb03-keyness_logodds_late_vs_early.png"
plt.savefig(fig_path, dpi=200)
plt.show()

df_key.head(200).to_csv(OUTPUT_DIR / "tables" / "nb03-keyness_logodds_late_vs_early_top200.csv", index=False)
print("Saved:", fig_path.name)
df_key.head(10)

## Bootstrap confidence intervals for time-bin comparisons

The descriptive comparisons above show how concept frequencies vary across `time_bin`, but point estimates alone can encourage overconfident interpretation. In a diachronic corpus, periods often differ in the number of documents, the amount of text, and the internal heterogeneity of the material they contain. As a result, apparent differences between bins may reflect not only substantive historical variation, but also sampling variability within the corpus.

To address this, we add a simple **bootstrap confidence interval** to each period-level estimate. The bootstrap resamples observations within each `time_bin`, recomputes the statistic many times, and uses the resulting empirical distribution to approximate an uncertainty range. Here, the goal is not formal hypothesis testing, nor to claim a definitive measure of historical change, but to provide a practical estimate of how stable the observed pattern appears under repeated resampling.

Methodologically, this matters because it disciplines interpretation. If a difference between periods remains visible across bootstrap resamples, we can treat it as more robust. If it varies substantially, we should be more cautious in treating the observed trend as historically meaningful. In that sense, the bootstrap functions here as a lightweight robustness check on our descriptive time-bin comparisons.

In [ ]:
# ------------------------------------------------------------
# Bootstrap confidence intervals for concept frequency by time bin
# Resamples documents/chunks within each time bin
# ------------------------------------------------------------

TARGET_CONCEPT = "reason"
N_BOOT = 1000
RANDOM_STATE = 42


def bin_start(label) -> int:
    s = str(label)
    m = re.search(r"-?\d+", s.replace("–", "-"))
    return int(m.group(0)) if m else 10**9


def count_term(text: str, term: str) -> int:
    pattern = rf"\b{re.escape(term.lower())}\b"
    return len(re.findall(pattern, str(text).lower()))


boot_df = df.dropna(subset=["time_bin", "text"]).copy()
boot_df["time_bin"] = boot_df["time_bin"].astype(str)
boot_df["n_tokens"] = boot_df["text"].astype(str).str.split().str.len()
boot_df["term_count"] = boot_df["text"].map(lambda x: count_term(x, TARGET_CONCEPT))

time_order = sorted(boot_df["time_bin"].unique().tolist(), key=bin_start)
rng = np.random.default_rng(RANDOM_STATE)
rows = []

for bin_label, sub in boot_df.groupby("time_bin"):
    counts = sub["term_count"].to_numpy()
    tokens = sub["n_tokens"].to_numpy()
    n = len(sub)

    observed = counts.sum() / tokens.sum() * 1_000_000 if tokens.sum() else np.nan

    boots = np.empty(N_BOOT, dtype=float)
    for b in range(N_BOOT):
        idx = rng.integers(0, n, size=n)
        boots[b] = counts[idx].sum() / tokens[idx].sum() * 1_000_000

    rows.append({
        "time_bin": str(bin_label),
        "rate_per_million": observed,
        "ci_low": np.percentile(boots, 2.5),
        "ci_high": np.percentile(boots, 97.5),
        "n_docs_or_chunks": n,
    })

boot_concept_ci = pd.DataFrame(rows).sort_values("time_bin", key=lambda s: s.map(bin_start))
display(boot_concept_ci)

plt.figure(figsize=(10, 5))
plt.plot(boot_concept_ci["time_bin"], boot_concept_ci["rate_per_million"], marker="o")
plt.fill_between(
    boot_concept_ci["time_bin"],
    boot_concept_ci["ci_low"],
    boot_concept_ci["ci_high"],
    alpha=0.25,
)
plt.title(f'Bootstrap CI for "{TARGET_CONCEPT}" frequency by time bin')
plt.xlabel("Time bin")
plt.ylabel("Occurrences per 1,000,000 tokens")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## Saved outputs (for the next notebook)

This notebook writes reusable artifacts to:

- `cache/session04/`:
  - `X_counts.npz` (sparse document–term matrix)
  - `vocab.json`, `vectorizer.joblib`, `doc_index.csv`
- `corpus_analysis/session04/`:
  - figures (`*.png`)
  - tables (`*.csv`)

These files let later notebooks load counts and trend tables without re-vectorizing the full corpus.

**Next step (Notebook 04 / Session 5):** lexical diversity, collocations/association, keyness (more formal), and a first TF–IDF vs embeddings comparison for exploratory “concept neighborhoods.”

```mermaid
flowchart TB
    A0["00<br/>Bootcamp"] --> P1

    subgraph P1["Part I — Corpus building and analysis"]
        direction LR
        A1a["01a<br/>Corpus metadata"] --> A1b["01b<br/>Corpus building"] --> A2["02<br/>Preprocessing"] --> A3["03<br/>Distributions + time"] --> A4a["04a<br/>Lexical exploration"] --> A4b["04b<br/>Embedding"]
    end

    subgraph P2["Part II — Linguistic annotations"]
        direction LR
        A5a["05a<br/>spaCy annotation"] --> A5b["05b<br/>Relation extraction"] --> A6a["06a<br/>NER"] --> A6b["06b<br/>Custom NER"]
    end

    subgraph P3["Part III — Representations"]
        direction LR
        A7["07<br/>BoW + TF-IDF"] --> A8a["08a<br/>Embeddings"] --> A8b["08b<br/>Transformers"]
    end

    subgraph P4["Part IV — Models and interpretation"]
        direction LR
        A9["09<br/>Classification"] --> A10["10<br/>Custom NER training"] --> A11["11<br/>Topic modeling"] --> A12["12<br/>Semantic shift"]
    end

    P1 --> P2
    P2 --> P3
    P3 --> P4

    classDef start fill:#f3f0ff,stroke:#6f42c1,stroke-width:1.5px,color:#111;
    classDef prep fill:#eef7ff,stroke:#1f77b4,stroke-width:1.5px,color:#111;
    classDef annot fill:#eefaf0,stroke:#2ca02c,stroke-width:1.5px,color:#111;
    classDef repr fill:#fff7e6,stroke:#ff8c00,stroke-width:1.5px,color:#111;
    classDef model fill:#fff0f0,stroke:#d62728,stroke-width:1.5px,color:#111;

    classDef highlight fill:#fff3b0,stroke:#f5a623,stroke-width:4px,color:#111;

    class A1a,A1b,A2,A3,A4a,A4b prep;
    class A5a,A5b,A6a,A6b annot;
    class A7,A8a,A8b repr;
    class A9,A10,A11,A12 model;

    class A3 highlight;
```